# proto-REINFORCE on the Golden Goose, part 2: the forward pass

A goose lays a golden egg, worth \$10, in 80% of the weeks, and a butcher pays \$50 for it. Every week the
farmer chooses: KEEP the goose, or SELL it. An episode is a quarter, 13 weeks, from GOOSE, and its return is the money
of its 13 weeks added up.

The forward pass judges a row of preferences by the average return of 1,000 episodes of its policy. Part 1 builds an
episode and its return; here they are given, finished, in a folded cell. This notebook does not need part 1.

- `play_episode(probabilities)`: one episode of 13 weeks with the policy `probabilities`, a row
  `[KEEP's probability, SELL's probability]`; gives the reward of every week, in a list.
- `episode_return(rewards)`: the rewards of the weeks added up.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one
only if you are still stuck. After each task, a check cell tells you whether your code works. The luck of the
goose is in the kit: your code never deals with dice.

Start with the folded cells below.

In [ ]:
#@title The goose's kit (run me first) { display-mode: "form" }
# The goose's kit. Run this cell first; you do not need to read it.
import inspect as _kit_inspect
import random                   # Python's random numbers: random.randint(0, 1) gives 0 or 1
import random as _kit_random
from math import exp            # e to the power: exp(0.5) is 1.6487...

GOOSE, SOLD = 0, 1              # the states
KEEP, SELL = 0, 1               # the actions
WEEKS = 13                      # an episode lasts a quarter: 13 weeks
EPISODES = 1000                 # an average takes 1,000 episodes

_KIT_FIRST_LUCK = 1
_KIT_LUCK_BASE = 1000
_KIT_PICKS = 1
_KIT = {"seed": _KIT_FIRST_LUCK, "luck": _kit_random.Random(_KIT_FIRST_LUCK), "update": 0, "spins": 0,
        "forced": None}
_KIT_NAMES = {"states": ["GOOSE", "SOLD"], "actions": ["KEEP", "SELL"]}


def spin():
    """The ball of a roulette: a number between 0 and 1, every number as likely as any other."""
    _KIT["spins"] = _KIT["spins"] + 1
    if _KIT["forced"] is not None:
        return _KIT["forced"]
    return _KIT["luck"].random()


def transition(state, action):
    """One week of the goose. Gives the state next week and the week's reward:
        next_state, reward = transition(GOOSE, KEEP)"""
    ball = spin()
    if state == SOLD:
        return SOLD, 0                  # a sold goose brings nothing more, whatever the action
    if action == SELL:
        return SOLD, 50                 # the butcher pays $50, and the goose is SOLD
    if ball < 0.8:
        return GOOSE, 10                # a golden egg, $10, and the goose stays
    return GOOSE, 0                     # no egg, $0, and the goose stays


def softmax(preferences):
    """A row of preferences to a row of probabilities: e to the power of each preference, divided by their total."""
    largest = max(preferences)
    powers = [exp(x - largest) for x in preferences]
    total = sum(powers)
    return [p / total for p in powers]


def same_luck():
    """The luck starts again from its beginning: every average meets the same weeks."""
    _KIT["luck"] = _kit_random.Random(_KIT["seed"])


def new_luck():
    """New episodes for the next update of proto-REINFORCE: the luck of seed 1000 plus the number of the update."""
    _KIT["update"] = _KIT["update"] + 1
    _KIT["seed"] = _KIT_LUCK_BASE + _KIT["update"]
    same_luck()


def start_loop():
    """The luck of the loop from its beginning, and the same picks of random.randint for everyone."""
    _kit_set_luck(_KIT_FIRST_LUCK)
    random.seed(_KIT_PICKS)


def _kit_set_luck(seed, update=0):
    _KIT["seed"] = seed
    _KIT["update"] = update
    same_luck()


def copy_preferences(preferences):
    """A copy of a row of preferences: changing the copy leaves the preferences as they are."""
    return list(preferences)


def _kit_money(x):
    return ("$" if x >= 0 else "-$") + format(abs(x), ",.2f")


def plot_learning_curve(points, best=None):
    """The learning curve: the average return before the first update, then after each update."""
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 3.5))
    plt.plot(range(len(points)), points, color="#C8102E", linewidth=2, label="average return of your preferences")
    if best is not None:
        plt.axhline(best, color="#333333", linestyle="--", linewidth=1, label="always KEEP")
    plt.xlabel("update")
    plt.ylabel("average return ($)")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()


def _kit_gaps(fn):
    """How many lines of the function still hold a gap, ..."""
    code_of = getattr(fn, "__code__", None)
    if code_of is None or Ellipsis not in code_of.co_consts:
        return 0
    try:
        src = _kit_inspect.getsource(fn)
    except (OSError, TypeError):
        return 1
    n = 0
    for line in src.splitlines():
        code_part = line.split("#")[0]
        if "..." in code_part:
            n = n + 1
    return n


def play(fn, *args):
    """Calls one of your functions once its check passes; otherwise prints one calm line."""
    for a in args:
        if a is None:
            print("⏳ %s waits for a value that an earlier cell has not given yet." % fn.__name__)
            return None
    try:
        ok = _KIT_CHECKS[fn.__name__](fn, quiet=True)
    except NameError:
        print("⏳ run the folded cell of the checks first.")
        return None
    except KeyError:
        return fn(*args)
    if not ok:
        print("⏳ %s is not ready yet: its check says what is missing." % fn.__name__)
        return None
    return fn(*args)


print("✅ The kit is ready: GOOSE, SOLD, KEEP, SELL, WEEKS, EPISODES, spin, transition, softmax, same_luck, new_luck.")

In [ ]:
#@title The functions of part 1, given (run me too) { display-mode: "form" }
# The functions of part 1, given. You do not need to read this cell.

# Given: the ends of the slots of a roulette
def make_roulette(probabilities):
    # probabilities: a row of probabilities, one per action. Gives the ends of the slots, a running total: [0.3, 1.0].
    ends = []
    running = 0                             # where the slots so far end: nowhere yet
    for probability in probabilities:
        running = running + probability     # this slot ends one probability further
        ends.append(running)                # remember where this slot ends
    return ends


# Given: the slot where the ball lands
def where_it_lands(ends, ball):
    # ends: the ends of the slots, from make_roulette. ball: a number between 0 and 1.
    # Gives the number of the first slot whose end is above the ball: 0 for KEEP, 1 for SELL.
    for slot in range(len(ends)):
        if ball < ends[slot]:
            return slot
    return len(ends) - 1                # a ball at the very end lands in the last slot


# Given: an action at random, with the probabilities of the policy
def pick_action(probabilities):
    # probabilities: the policy, a row of probabilities [KEEP, SELL]. Gives KEEP (0) or SELL (1), sampled at random.
    ends = make_roulette(probabilities)             # the roulette of the probabilities
    ball = spin()                       # the ball
    return where_it_lands(ends, ball)   # where the ball lands


# Given: one episode of the goose
def play_episode(probabilities):
    # probabilities: the policy. Plays WEEKS weeks from GOOSE and gives the rewards, one per week, in a list.
    rewards = []
    state = GOOSE
    for week in range(WEEKS):
        action = pick_action(probabilities)         # an action sampled from the policy
        state, reward = transition(state, action)   # the goose's week
        rewards.append(reward)                      # remember the week's reward
    return rewards


# Given: the return of an episode
def episode_return(rewards):
    # rewards: the reward of every week of an episode. Gives G1: every week's reward counts in full.
    G1 = 0
    for reward in rewards:
        G1 = G1 + reward                # add the week's reward
    return G1


print('✅ The functions of part 1 are ready: pick_action, play_episode, episode_return.')

In [ ]:
#@title The checks (run me too) { display-mode: "form" }
# The checks of the notebook. Run this cell too; you do not need to read it.
# A check prints one short message and never stops the notebook.

def _ref_make_roulette(probabilities):
    ends, running = [], 0
    for p in probabilities:
        running = running + p
        ends.append(running)
    return ends


def _ref_where_it_lands(ends, ball):
    for slot in range(len(ends)):
        if ball < ends[slot]:
            return slot
    return len(ends) - 1


def _ref_pick_action(probabilities):
    return _ref_where_it_lands(_ref_make_roulette(probabilities), spin())


def _ref_play_episode(probabilities):
    rewards, state = [], GOOSE
    for week in range(WEEKS):
        action = _ref_pick_action(probabilities)
        state, reward = transition(state, action)
        rewards.append(reward)
    return rewards


def _ref_average_return(preferences):
    same_luck()
    probabilities = softmax(preferences)
    total = 0
    for episode in range(EPISODES):
        total = total + sum(_ref_play_episode(probabilities))
    return round(total / EPISODES, 2)


def _ref_slope(preferences, action, hop):
    before = _ref_average_return(preferences)
    hopped = list(preferences)
    hopped[action] = hopped[action] + hop
    return round((_ref_average_return(hopped) - before) / hop)


def _kit_try(name, fn, args, say):
    """Calls fn calmly: (True, result), or (False, None) after one line."""
    try:
        return True, fn(*args)
    except Exception as e:
        say("❌ %s stops with an error: %s: %s" % (name, type(e).__name__, e))
        return False, None


def _kit_ready(name, fn, task, say, needs=()):
    if not callable(fn):
        say("⏳ %s is not a function yet: run the cell of task %s first." % (name, task))
        return False
    gaps = _kit_gaps(fn)
    if gaps > 0:
        say("⏳ %s is not finished yet: replace every ... of task %s with your code (%d left)." % (name, task, gaps))
        return False
    for other in needs:
        f = globals().get(other)
        if f is None or not _KIT_CHECKS[other](f, quiet=True):
            say("⏳ %s calls %s: finish %s first, its check says what is missing." % (name, other, other))
            return False
    return True


def _kit_check_pick_action(pick_action, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("pick_action", pick_action, "A", say):
        return False
    for probabilities, ball, want in (([0.3, 0.7], 0.1, KEEP), ([0.3, 0.7], 0.5, SELL), ([0.9, 0.1], 0.89, KEEP),
                                      ([0.9, 0.1], 0.95, SELL), ([0.3, 0.7], 0.3, SELL)):
        _KIT["forced"], _KIT["spins"] = ball, 0
        try:
            ok, got = _kit_try("pick_action(%s)" % probabilities, pick_action, (probabilities,), say)
        finally:
            _KIT["forced"] = None
        if not ok:
            return False
        if _KIT["spins"] != 1:
            say("❌ pick_action spins %d balls. It spins one: ball = spin(), once." % _KIT["spins"])
            return False
        if got != want:
            say("❌ pick_action(%s) with the ball at %s gives %r. The ends are %s, so the ball lands in %s's slot, %d."
                % (probabilities, ball, got, _ref_make_roulette(probabilities), _KIT_NAMES["actions"][want], want))
            return False
    say("✅ pick_action picks the slot where the ball lands in all 5 tests. Task A is done.")
    return True


def _kit_check_play_episode(play_episode, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("play_episode", play_episode, "B", say, ("pick_action",)):
        return False
    for probabilities, seed in (([0.5, 0.5], 7), ([0.9, 0.1], 8), ([1.0, 0.0], 9)):
        _kit_set_luck(seed)
        ok, got = _kit_try("play_episode(%s)" % probabilities, play_episode, (probabilities,), say)
        if not ok:
            return False
        _kit_set_luck(seed)
        want = _ref_play_episode(probabilities)
        if not isinstance(got, list):
            say("❌ play_episode gives %r. It gives the list of the episode's rewards, one per week." % (got,))
            return False
        if len(got) != WEEKS:
            say("❌ play_episode gives %d rewards. An episode has WEEKS = 13 weeks, one reward each, also once the "
                "goose is SOLD." % len(got))
            return False
        if got != want:
            say("❌ play_episode(%s) gives %s on the luck of the check; it should give %s. Each week: the action "
                "with pick_action(probabilities), then transition(state, action), from the state the week before left."
                % (probabilities, got, want))
            return False
    _kit_set_luck(_KIT_FIRST_LUCK)
    say("✅ play_episode plays 13 weeks in all 3 tests. Task B is done.")
    return True


def _kit_check_episode_return(episode_return, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("episode_return", episode_return, "C", say):
        return False
    for rewards in ([10, 10, 0, 10, 50, 0, 0, 0, 0, 0, 0, 0, 0], [50] + [0] * 12, [10] * 13, [0, 10, 10]):
        ok, got = _kit_try("episode_return", episode_return, (rewards,), say)
        if not ok:
            return False
        if got != sum(rewards):
            say("❌ episode_return(%s) gives %r. Every week counts in full: the total is %s."
                % (rewards, got, sum(rewards)))
            return False
    say("✅ episode_return adds up the rewards in all 4 tests. Task C is done.")
    return True


def _kit_check_average_return(average_return, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("average_return", average_return, "A", say, ("play_episode", "episode_return")):
        return False
    for preferences in ([0.0, 0.0], [2.0, -1.0]):
        _kit_set_luck(_KIT_FIRST_LUCK + 1)
        _KIT["luck"] = _kit_random.Random(12345)          # a luck the function must replace with same_luck()
        ok, got = _kit_try("average_return(%s)" % preferences, average_return, (preferences,), say)
        if not ok:
            return False
        want = _ref_average_return(preferences)
        if got != want:
            say("❌ average_return(%s) gives %r; it should give %s. Its lines: same_luck(), the probabilities with "
                "softmax(preferences), then EPISODES episodes, each one's G1 added to the total, and the total "
                "divided by EPISODES." % (preferences, got, want))
            _kit_set_luck(_KIT_FIRST_LUCK)
            return False
    _kit_set_luck(_KIT_FIRST_LUCK)
    say("✅ average_return gives the average return of 1,000 episodes in both tests. Task A is done.")
    return True


def _kit_check_pick_preference(pick_preference, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("pick_preference", pick_preference, "E", say):
        return False
    saved = random.getstate()
    seen = set()
    try:
        for i in range(200):
            ok, got = _kit_try("pick_preference()", pick_preference, (), say)
            if not ok:
                return False
            if got not in (0, 1):
                say("❌ pick_preference() gives %r. It gives the action of a preference: 0 for KEEP or 1 for SELL."
                    % (got,))
                return False
            seen.add(got)
    finally:
        random.setstate(saved)
    if seen != {0, 1}:
        say("❌ pick_preference() gave only %s in 200 picks. It picks KEEP or SELL at random: random.randint(0, 1)."
            % _KIT_NAMES["actions"][seen.pop()])
        return False
    say("✅ pick_preference picks KEEP and SELL. Task E is done.")
    return True


def _kit_check_slope(slope, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("slope", slope, "F", say, ("average_return",)):
        return False
    for preferences, action in (([-0.5, 0.5], KEEP), ([-0.5, 0.5], SELL), ([1.0, 0.0], KEEP)):
        before = list(preferences)
        _kit_set_luck(_KIT_FIRST_LUCK)
        ok, got = _kit_try("slope(%s, %s, 0.01)" % (preferences, _KIT_NAMES["actions"][action]), slope,
                           (preferences, action, 0.01), say)
        if not ok:
            return False
        if preferences != before:
            say("❌ slope changed the preferences themselves, to %s. The hop goes into a copy: "
                "hopped = copy_preferences(preferences)." % preferences)
            return False
        _kit_set_luck(_KIT_FIRST_LUCK)
        want = _ref_slope(preferences, action, 0.01)
        if got != want:
            say("❌ slope(%s, %s, 0.01) gives %r; it should give %s: Before, After with the preference a hop higher "
                "in a copy, then (After - Before) / hop, rounded." % (preferences, _KIT_NAMES["actions"][action],
                                                                       got, want))
            return False
    _kit_set_luck(_KIT_FIRST_LUCK)
    say("✅ slope gives nabla in all 3 tests. Task F is done.")
    return True


def _kit_check_update_preference(update_preference, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("update_preference", update_preference, "G", say):
        return False
    for preferences, action, nabla, alpha in (([-0.5, 0.5], KEEP, 37, 0.01), ([-0.5, 0.5], SELL, -12, 0.01),
                                              ([1.0, 2.0], KEEP, 0, 0.01)):
        row = list(preferences)
        ok, got = _kit_try("update_preference", update_preference, (row, action, nabla, alpha), say)
        if not ok:
            return False
        want = round(preferences[action] + alpha * nabla, 2)
        if row[action] != want:
            say("❌ update_preference(%s, %s, %s, %s) leaves the preference at %r; it should be %s: theta + alpha * "
                "nabla." % (preferences, _KIT_NAMES["actions"][action], nabla, alpha, row[action], want))
            return False
    say("✅ update_preference moves the preference in all 3 tests. Task G is done.")
    return True


def _kit_check_proto_reinforce(proto_reinforce, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("proto_reinforce", proto_reinforce, "H", say, ("pick_preference", "slope", "update_preference")):
        return False
    preferences = [-0.5, 0.5]
    start_loop()
    ok, got = _kit_try("proto_reinforce", proto_reinforce, (preferences, 3, 0.01, 0.01), say)
    if not ok:
        return False
    start_loop()
    want_preferences = [-0.5, 0.5]
    want = [_ref_average_return(want_preferences)]
    for u in range(3):
        new_luck()
        action = random.randint(0, 1)
        nabla = _ref_slope(want_preferences, action, 0.01)
        want_preferences[action] = round(want_preferences[action] + 0.01 * nabla, 2)
        want.append(_ref_average_return(want_preferences))
    start_loop()
    _kit_set_luck(_KIT_FIRST_LUCK)
    if not isinstance(got, list) or len(got) != 4:
        say("❌ proto_reinforce(preferences, 3, ...) gives %r. It gives the points of the learning curve: the first "
            "point, then one after each of the 3 updates, 4 in all." % (got,))
        return False
    if preferences != want_preferences or got != want:
        say("❌ 3 updates from [-0.5, 0.5] give the preferences %s and the points %s; they should give %s and %s. "
            "Each update: new_luck(), pick_preference(), slope, update_preference, then the point "
            "average_return(preferences)." % (preferences, got, want_preferences, want))
        return False
    say("✅ proto_reinforce plays 3 updates as it should. Task H is done.")
    return True


_KIT_CHECKS = {
    "pick_action": _kit_check_pick_action,
    "play_episode": _kit_check_play_episode,
    "episode_return": _kit_check_episode_return,
    "average_return": _kit_check_average_return,
    "pick_preference": _kit_check_pick_preference,
    "slope": _kit_check_slope,
    "update_preference": _kit_check_update_preference,
    "proto_reinforce": _kit_check_proto_reinforce,
}


def check_pick_action(pick_action):
    _kit_check_pick_action(pick_action)


def check_play_episode(play_episode):
    _kit_check_play_episode(play_episode)


def check_episode_return(episode_return):
    _kit_check_episode_return(episode_return)


def check_average_return(average_return):
    _kit_check_average_return(average_return)


def check_pick_preference(pick_preference):
    _kit_check_pick_preference(pick_preference)


def check_slope(slope):
    _kit_check_slope(slope)


def check_update_preference(update_preference):
    _kit_check_update_preference(update_preference)


def check_proto_reinforce(proto_reinforce):
    _kit_check_proto_reinforce(proto_reinforce)


print("✅ The checks are ready.")

## 1. Preferences and softmax

proto-REINFORCE does not change probabilities directly. It changes preferences, one per action, and softmax turns
them into probabilities: the higher the preference, the higher the probability, and the probabilities add up to 1.
The kit holds `softmax(preferences)`.

In [ ]:
preferences = [-0.5, 0.5]           # KEEP's preference, SELL's preference
print("preferences  ", preferences)
print("probabilities", [round(p, 2) for p in softmax(preferences)])

**Reading the result.** SELL's preference is 1 above KEEP's, so SELL is the likelier action: KEEP's probability is
0.27, SELL's 0.73. These preferences are the start of the loop in part 3, the backward pass.

## 2. The average return $\bar{G}_1$ of a row of preferences

The return of an episode is written $G_1$. One episode says little: luck decides much of it. The forward pass judges a row of preferences by the
average return $\bar{G}_1$ of `EPISODES` = 1,000 episodes of its policy. `same_luck()` starts the luck from its
beginning, so every row of preferences meets the same weeks and two rows are compared on the same luck.

### 🎯 Task A. The forward pass

Write `average_return(preferences)`: the probabilities with `softmax`, then `EPISODES` episodes with
`play_episode`, each episode's $G_1$ with `episode_return` added to the total.

In [ ]:
# ✅ SOLUTION, task A: the forward pass, the average return of a row of preferences
def average_return(preferences):
    # preferences: [KEEP's preference, SELL's preference]. Gives the average return of EPISODES episodes of the
    # policy of the preferences, in cents.
    same_luck()                         # every average plays the same luck: the same episodes for every row
    probabilities = softmax(preferences)            # the probabilities of the preferences
    total = 0
    for episode in range(EPISODES):
        rewards = play_episode(probabilities)       # one episode
        total = total + episode_return(rewards)     # add the episode's return
    return round(total / EPISODES, 2)   # the average, in cents

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# probabilities = softmax(preferences).
# In the loop: one episode is play_episode(probabilities), its G1 is episode_return(rewards).

In [ ]:
#@title 💡 Hint 2 for task A (double-click to open) { display-mode: "form" }
#     probabilities = softmax(preferences)
#         rewards = play_episode(probabilities)
#         total = total + episode_return(rewards)

In [ ]:
check_average_return(average_return)

💭 **Think first.** Three rows of preferences: the start, a row that always KEEPs, a row that always SELLs. Which
earns the most over a quarter?

In [ ]:
print("the start      [-0.5, 0.5] ", play(average_return, [-0.5, 0.5]))
print("always KEEP    [10, -10]   ", play(average_return, [10, -10]))
print("always SELL    [-10, 10]   ", play(average_return, [-10, 10]))

**Reading the result.** Always SELL earns \$50.00: the butcher's money, at once. Always KEEP earns \$103.61, about
13 weeks of \$8 on average. The start, which sells in most weeks, earns \$52.99. The money at once loses to the
eggs over a quarter.

## What you built

- **The forward pass**: `average_return`, the average return of 1,000 episodes of the policy of a row of
  preferences, all on the same luck.

Next, part 3 builds the backward pass: [proto-REINFORCE on the Golden Goose, part 3: the backward pass](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/05_cx_golden_goose_3_backward_pass_solutions.ipynb).